# Contract Intelligence Platform — Data Pipeline

**What this notebook does:**
1. Mounts Google Drive and installs dependencies
2. Defines all configuration inline (no separate config.py needed)
3. Downloads and processes **three datasets**:
   - **CUAD** — 510 contracts, 41 clause types (~3,500 labelled clauses)
   - **LEDGAR** — ~80,000 contract provisions, 100 provision types
   - **MAUD** — ~39,000 M&A deal excerpts, 14 deal-point types
4. Unifies all data under a single **100-type clause taxonomy**
5. Saves everything to SQLite (~120,000 clauses total)
6. Generates stratified train / val / test CSV splits
7. Downloads all output files to your local machine

**EDGAR enrichment (optional — run locally, not in Colab):**
After downloading the outputs, run on your local machine:
```
python src/data_pipeline.py --mode edgar --limit 5000
```
Then review `data/processed/review_queue.csv` and `edgar_new_clause_types.csv`,
re-run `--mode split`, and upload new CSVs to Colab for retraining.

**Run all cells top to bottom. Do not skip any cell.**

**Outputs to place locally at** `data/processed/`:
- `contracts.db`
- `train.csv`
- `val.csv`
- `test.csv`

## Step 1 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'pdfplumber', 'sqlalchemy',
    'pandas', 'numpy', 'scikit-learn', 'tqdm', 'loguru', 'torch'
])
print('Dependencies installed.')

## Step 2 — Configuration & Directory Setup

In [ ]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT_DIR      = Path('/content/drive/MyDrive/contract-intelligence-platform')
DATA_DIR      = ROOT_DIR / 'data'
RAW_DIR       = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
CUAD_DIR      = DATA_DIR / 'cuad'
LEDGAR_DIR    = DATA_DIR / 'ledgar'
MAUD_DIR      = DATA_DIR / 'maud'
EDGAR_DIR     = DATA_DIR / 'edgar'
EDGAR_RAW_DIR = EDGAR_DIR / 'raw'
MODELS_DIR    = ROOT_DIR / 'models'
LOGS_DIR      = ROOT_DIR / 'logs'

DB_PATH = PROCESSED_DIR / 'contracts.db'
DB_URL  = f'sqlite:///{DB_PATH}'

# ── HuggingFace identifiers ────────────────────────────────────────────────
LEGAL_BERT_MODEL  = 'nlpaueb/legal-bert-base-uncased'
CUAD_DATASET_ID   = 'theatticusproject/cuad-qa'
LEDGAR_DATASET_ID = 'coastalcph/lex_glue'
MAUD_DATASET_ID   = 'theatticusproject/maud'

# ── EDGAR settings ─────────────────────────────────────────────────────────
EDGAR_AUTO_LABEL_CONFIDENCE = 0.70
EDGAR_DOWNLOAD_LIMIT        = 5000
EDGAR_CLUSTER_N             = 20
EDGAR_REVIEW_PATH           = PROCESSED_DIR / 'review_queue.csv'
EDGAR_NEW_TYPES_PATH        = PROCESSED_DIR / 'edgar_new_clause_types.csv'

# ── Unified clause taxonomy (100 types) ───────────────────────────────────
CUAD_CLAUSE_TYPES = [
    # CUAD core (41)
    'Competitive Restriction Exception', 'Non-Compete', 'Exclusivity',
    'No-Solicit Of Customers', 'No-Solicit Of Employees', 'Non-Disparagement',
    'Termination For Convenience', 'Rofr/Rofo/Rofn', 'Change Of Control',
    'Anti-Assignment', 'Revenue/Profit Sharing', 'Price Restrictions',
    'Minimum Commitment', 'Volume Restriction', 'IP Ownership Assignment',
    'Joint IP Ownership', 'License Grant', 'Non-Transferable License',
    'Affiliate License-Licensor', 'Affiliate License-Licensee',
    'Unlimited/All-You-Can-Eat-License', 'Irrevocable Or Perpetual License',
    'Source Code Escrow', 'Post-Termination Services', 'Audit Rights',
    'Uncapped Liability', 'Cap On Liability', 'Liquidated Damages',
    'Warranty Duration', 'Insurance', 'Covenant Not To Sue',
    'Third Party Beneficiary', 'Most Favored Nation', 'Governing Law',
    'Dispute Resolution', 'Limitations Of Liability', 'Indemnification',
    'Agreement Date', 'Effective Date', 'Expiration Date', 'Renewal Term',
    # New from LEDGAR (32)
    'Confidentiality', 'Amendments', 'Anti-Corruption Laws',
    'Approvals And Consents', 'Authority', 'Compliance With Laws',
    'Consent To Jurisdiction', 'Definitions', 'Enforceability',
    'Entire Agreements', 'Expenses', 'Force Majeure', 'Further Assurances',
    'Interests', 'Liens And Encumbrances', 'Limitations Of Remedies',
    'Non-Waiver', 'Notices', 'Organization And Existence', 'Payments',
    'Representations And Warranties', 'Sanctions', 'Securities Law Compliance',
    'Severability', 'Specific Performance', 'Successors And Assigns',
    'Survival', 'Taxes And Withholding', 'Trade Controls',
    'Transactions With Affiliates', 'Waiver Of Jury Trial', 'Waivers',
    # New from MAUD (14)
    'No-Shop', 'Material Adverse Effect', 'Hell-Or-High-Water',
    'Termination Fee', 'Reverse Termination Fee', 'Matching Rights',
    'Superior Offer Definition', 'Fiduciary Exception',
    'Antitrust Efforts Standard', 'Operating Covenants',
    'Expense Reimbursement', 'Intervening Event Definition',
    'Board Recommendation Change', 'Tail Period',
    # Additional commercial provisions (13)
    'Data Protection And Privacy', 'Employment And Benefits', 'Capitalization',
    'Publicity And Announcements', 'Non-Solicitation General', 'Step-In Rights',
    'Benchmarking Rights', 'Electronic Signatures And Counterparts',
    'Service Level Agreement', 'Subcontracting', 'Assignment Of Receivables',
    'Performance Bonds', 'Escrow And Holdback',
]
NUM_CLAUSE_TYPES = len(CUAD_CLAUSE_TYPES)  # 100

# ── Hyperparameters ────────────────────────────────────────────────────────
CLAUSE_MIN_TOKENS = 10
CLAUSE_MAX_TOKENS = 512
MAX_PAGES         = 100
TRAIN_RATIO       = 0.80
VAL_RATIO         = 0.10
TEST_RATIO        = 0.10
RANDOM_SEED       = 42
UNDERSAMPLE_MAX_PER_CLASS = None  # cap per class in training set; None to disable

# ── Create all directories ─────────────────────────────────────────────────
for _d in [RAW_DIR, PROCESSED_DIR, CUAD_DIR, LEDGAR_DIR, MAUD_DIR,
           EDGAR_DIR, EDGAR_RAW_DIR, MODELS_DIR, LOGS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print('Configuration ready.')
print(f'  ROOT_DIR      : {ROOT_DIR}')
print(f'  PROCESSED_DIR : {PROCESSED_DIR}')
print(f'  Clause types  : {NUM_CLAUSE_TYPES}')

## Step 3 — Imports

In [ ]:
import hashlib
import re
import sys
from contextlib import contextmanager
from datetime import datetime, timezone
from typing import Dict, Generator, List, Optional, Tuple

import numpy as np
import pandas as pd
import pdfplumber
import torch
from datasets import load_dataset
from loguru import logger
from sklearn.model_selection import train_test_split
from sqlalchemy import (
    Boolean, Column, DateTime, Float, ForeignKey,
    Integer, String, Text, create_engine, event, text as sa_text,
)
from sqlalchemy.orm import DeclarativeBase, Session, relationship, sessionmaker
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

logger.remove()
logger.add(str(LOGS_DIR / 'data_pipeline.log'), rotation='10 MB', level='DEBUG')
logger.add(sys.stderr, level='INFO')
print('Imports complete.')

## Step 4 — Database Setup (SQLite / SQLAlchemy)

In [ ]:
_engine = create_engine(DB_URL, connect_args={'check_same_thread': False}, echo=False)

@event.listens_for(_engine, 'connect')
def _set_pragma(dbapi_conn, _):
    cur = dbapi_conn.cursor()
    cur.execute('PRAGMA journal_mode=WAL')
    cur.execute('PRAGMA foreign_keys=ON')
    cur.close()

SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=_engine)

class Base(DeclarativeBase):
    pass

class Contract(Base):
    __tablename__ = 'contracts'
    contract_id = Column(String(64), primary_key=True, index=True)
    filename    = Column(String(512), nullable=False)
    source      = Column(String(64),  nullable=False, default='upload')
    page_count  = Column(Integer, nullable=True)
    created_at  = Column(DateTime, default=lambda: datetime.now(timezone.utc), nullable=False)
    clauses     = relationship('Clause', back_populates='contract', cascade='all, delete-orphan')

class Clause(Base):
    __tablename__ = 'clauses'
    clause_id             = Column(String(64), primary_key=True, index=True)
    contract_id           = Column(String(64), ForeignKey('contracts.contract_id'), nullable=False, index=True)
    clause_text           = Column(Text,        nullable=False)
    clause_type           = Column(String(512), nullable=False, default='')
    party_a               = Column(String(256), nullable=True, default='')
    party_b               = Column(String(256), nullable=True, default='')
    source                = Column(String(64),  nullable=True, default='')
    anomaly_score         = Column(Float,   nullable=True)
    is_anomalous          = Column(Boolean, nullable=True, default=False)
    power_imbalance_score = Column(Float,   nullable=True)
    party_a_leverage      = Column(Float,   nullable=True)
    party_b_leverage      = Column(Float,   nullable=True)
    sentiment_score       = Column(Float,   nullable=True)
    modal_score           = Column(Float,   nullable=True)
    obligation_score      = Column(Float,   nullable=True)
    assertiveness_score   = Column(Float,   nullable=True)
    shap_plot_path        = Column(String(512), nullable=True)
    created_at            = Column(DateTime, default=lambda: datetime.now(timezone.utc), nullable=False)
    contract = relationship('Contract', back_populates='clauses')

def create_tables():
    Base.metadata.create_all(bind=_engine)

create_tables()
print(f'Database ready: {DB_PATH}')

## Step 5 — Pipeline Classes

In [ ]:
# ── PDF Ingester ───────────────────────────────────────────────────────────
class PDFIngester:
    def __init__(self, max_pages=MAX_PAGES):
        self.max_pages = max_pages

    def extract_text(self, pdf_path: Path) -> str:
        if not pdf_path.exists():
            raise FileNotFoundError(f'PDF not found: {pdf_path}')
        pages = []
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages[:self.max_pages]:
                t = page.extract_text(x_tolerance=2, y_tolerance=2)
                if t:
                    pages.append(t)
        full = self._clean('\n'.join(pages))
        if not full.strip():
            raise ValueError(f'No text in {pdf_path.name}')
        return full

    @staticmethod
    def _clean(t):
        t = re.sub(r'[^\x09\x0A\x0D\x20-\x7E\u00A0-\uFFFF]', ' ', t)
        t = re.sub(r'[ \t]{2,}', ' ', t)
        t = re.sub(r'\n{3,}', '\n\n', t)
        return t.strip()


# ── Clause Segmenter ──────────────────────────────────────────────────────
class ClauseSegmenter:
    PATTERNS = [
        re.compile(r'^(?:ARTICLE|SECTION|CLAUSE)\s+\d+',  re.MULTILINE),
        re.compile(r'^\d+\.\d*\s+[A-Z]',                  re.MULTILINE),
        re.compile(r'^\d+\.\s+[A-Z]',                     re.MULTILINE),
        re.compile(r'^[A-Z][A-Z\s]{4,}$',                 re.MULTILINE),
    ]

    def __init__(self, tokenizer=None, min_tokens=CLAUSE_MIN_TOKENS, max_tokens=CLAUSE_MAX_TOKENS):
        self.min_tokens = min_tokens
        self.max_tokens = max_tokens
        self.tokenizer  = tokenizer

    def segment(self, text: str) -> List[str]:
        clauses = self._by_headers(text)
        if len(clauses) < 5:
            clauses = [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]
        clauses = [re.sub(r'[ \t]{2,}', ' ', c).strip() for c in clauses]
        clauses = [c for c in clauses if len(c.split()) >= self.min_tokens]
        if self.tokenizer:
            clauses = [self._truncate(c) for c in clauses]
        return clauses

    def _by_headers(self, text):
        lines, segs, cur = text.split('\n'), [], []
        for line in lines:
            if line.strip() and any(p.match(line.strip()) for p in self.PATTERNS):
                if cur:
                    segs.append('\n'.join(cur).strip())
                cur = [line]
            else:
                cur.append(line)
        if cur:
            segs.append('\n'.join(cur).strip())
        return segs

    def _truncate(self, clause):
        ids = self.tokenizer(clause, max_length=self.max_tokens, truncation=True, return_tensors='pt')
        return self.tokenizer.decode(ids['input_ids'][0], skip_special_tokens=True)

print('PDFIngester and ClauseSegmenter defined.')

In [ ]:
# ── CUAD Processor ────────────────────────────────────────────────────────
class CUADProcessor:
    def __init__(self):
        self.clause_types = CUAD_CLAUSE_TYPES

    def load_and_process(self, cache_dir=CUAD_DIR) -> pd.DataFrame:
        logger.info('Loading CUAD dataset from HuggingFace...')
        dataset  = load_dataset(CUAD_DATASET_ID, cache_dir=str(cache_dir))
        all_items = list(dataset['train']) + list(dataset['test'])

        contract_map: Dict[str, List] = {}
        for item in tqdm(all_items, desc='Parsing CUAD'):
            name = item.get('title', '').strip()
            if not name:
                parts = item.get('id', '').split('__')
                name  = parts[0].strip() if parts else 'unknown'
            contract_map.setdefault(name or 'unknown', []).append(item)

        logger.info(f'Found {len(contract_map)} unique contracts')
        records = []

        for contract_name, items in tqdm(contract_map.items(), desc='Processing contracts'):
            contract_id = hashlib.md5(contract_name.encode()).hexdigest()[:12]
            seen: Dict[str, List[str]] = {}

            for item in items:
                ct = self._infer_clause_type(item)
                if ct is None:
                    continue
                answers = item.get('answers', {}) or {}
                texts   = answers.get('text', []) or []
                if isinstance(texts, str):
                    texts = [texts]
                for ans in texts:
                    if not isinstance(ans, str):
                        continue
                    ans = ans.strip()
                    if ans and len(ans.split()) >= CLAUSE_MIN_TOKENS:
                        seen.setdefault(ans, []).append(ct)

            for clause_text, ct_list in seen.items():
                clause_id = hashlib.md5(f'{contract_id}{clause_text}'.encode()).hexdigest()[:16]
                records.append({
                    'contract_id':  contract_id,
                    'clause_id':    clause_id,
                    'clause_text':  clause_text,
                    'clause_types': sorted(set(ct_list)),
                    'party_a':      self._party(clause_text, 'a'),
                    'party_b':      self._party(clause_text, 'b'),
                    'source':       'CUAD',
                })

        df = pd.DataFrame(records)
        logger.info(f'Processed {len(df)} clauses from {len(contract_map)} contracts')
        return df

    def _infer_clause_type(self, item) -> Optional[str]:
        raw_id   = item.get('id', '')
        question = item.get('question', '').strip().lower()
        if '__' in raw_id:
            parts = raw_id.split('__')
            if len(parts) >= 2:
                cand = parts[-1].replace('_', ' ').strip().lower()
                for ct in self.clause_types:
                    if ct.lower() == cand or ct.lower() in cand or cand in ct.lower():
                        return ct
        for ct in self.clause_types:
            if ct.lower() in question or question in ct.lower():
                return ct
        return None

    @staticmethod
    def _party(text, side):
        pats = (['(Company|Licensor|Seller|Vendor|Provider)', r'"(Company|Licensor|Seller|Vendor|Provider)"']
                if side == 'a' else
                ['(Counterparty|Licensee|Buyer|Customer)', r'"(Counterparty|Licensee|Buyer|Customer)"'])
        for p in pats:
            m = re.search(p, text)
            if m:
                return m.group(1)
        return ''

print('CUADProcessor defined.')

In [ ]:
# ── LEDGAR label → unified taxonomy map ───────────────────────────────────
_LEDGAR_LABEL_MAP = {
    'Adjustments': None, 'Agreements': None,
    'Amendments': 'Amendments',
    'Anti-Corruption Laws': 'Anti-Corruption Laws',
    'Approvals': 'Approvals And Consents',
    'Arbitration': 'Dispute Resolution',
    'Assignments': 'Anti-Assignment',
    'Authority': 'Authority',
    'Base Salary': 'Employment And Benefits',
    'Benefits': 'Employment And Benefits',
    'Capitalization': 'Capitalization',
    'Closings': None,
    'Compliance With Laws': 'Compliance With Laws',
    'Confidentiality': 'Confidentiality',
    'Consent To Jurisdiction': 'Consent To Jurisdiction',
    'Construction': 'Definitions',
    'Definitions': 'Definitions',
    'Disability': 'Employment And Benefits',
    'Effective Dates': 'Effective Date',
    'Employment': 'Employment And Benefits',
    'Enforceability': 'Enforceability',
    'Entire Agreements': 'Entire Agreements',
    'Erisa': 'Employment And Benefits',
    'Expenses': 'Expenses',
    'Further Assurances': 'Further Assurances',
    'General': None, 'Governing Laws': 'Governing Law',
    'Headings': None,
    'Indemnifications': 'Indemnification',
    'Insurances': 'Insurance',
    'Integration': 'Entire Agreements',
    'Intellectual Property': 'IP Ownership Assignment',
    'Interests': 'Interests',
    'Jurisdictions': 'Consent To Jurisdiction',
    'Liens': 'Liens And Encumbrances',
    'Limitations Of Remedies': 'Limitations Of Remedies',
    'Non-Waivers': 'Non-Waiver',
    'Notices': 'Notices',
    'No Third-Party Beneficiaries': 'Third Party Beneficiary',
    'Obligations': None,
    'Organization': 'Organization And Existence',
    'Payments': 'Payments',
    'Remedies': 'Limitations Of Remedies',
    'Representations': 'Representations And Warranties',
    'Sanctions': 'Sanctions',
    'Securities Laws': 'Securities Law Compliance',
    'Severability': 'Severability',
    'Specific Performance': 'Specific Performance',
    'Successors': 'Successors And Assigns',
    'Survival': 'Survival',
    'Taxes': 'Taxes And Withholding',
    'Terminations': 'Termination For Convenience',
    'Titles': None,
    'Trade Controls': 'Trade Controls',
    'Transactions With Affiliates': 'Transactions With Affiliates',
    'Transfers': 'Anti-Assignment',
    'Waiver Of Jury Trial': 'Waiver Of Jury Trial',
    'Waivers': 'Waivers',
    'Warranties': 'Representations And Warranties',
    'Withholding': 'Taxes And Withholding',
}


# ── LEDGAR Processor ──────────────────────────────────────────────────────
class LEDGARProcessor:
    def load_and_process(self, cache_dir=LEDGAR_DIR) -> pd.DataFrame:
        logger.info('Loading LEDGAR dataset from HuggingFace (lex_glue/ledgar)...')
        dataset = load_dataset(LEDGAR_DATASET_ID, 'ledgar', cache_dir=str(cache_dir))

        records = []
        skipped = 0

        for split_name in ('train', 'validation', 'test'):
            if split_name not in dataset:
                continue
            split = dataset[split_name]
            label_feature = split.features['label']

            for item in tqdm(split, desc=f'LEDGAR {split_name}'):
                text = (item.get('text') or '').strip()
                if not text or len(text.split()) < CLAUSE_MIN_TOKENS:
                    skipped += 1
                    continue

                ledgar_label  = label_feature.int2str(item['label'])
                unified_type  = _LEDGAR_LABEL_MAP.get(ledgar_label)
                if unified_type is None:
                    skipped += 1
                    continue

                clause_id   = hashlib.md5(f'LEDGAR{text}'.encode()).hexdigest()[:16]
                contract_id = hashlib.md5(f'LEDGAR_{item["label"]}'.encode()).hexdigest()[:12]
                records.append({
                    'contract_id':  contract_id,
                    'clause_id':    clause_id,
                    'clause_text':  text,
                    'clause_types': [unified_type],
                    'party_a': '', 'party_b': '',
                    'source': 'LEDGAR',
                })

        df = pd.DataFrame(records).drop_duplicates(subset='clause_id').reset_index(drop=True)
        logger.info(f'LEDGAR: {len(df)} clauses retained, {skipped} skipped')
        return df


# ── MAUD question → unified taxonomy map (priority-ordered) ───────────────
_MAUD_QUESTION_MAP = [
    ('reverse termination fee',  'Reverse Termination Fee'),
    ('termination fee',          'Termination Fee'),
    ('hell-or-high-water',       'Hell-Or-High-Water'),
    ('no-shop',                  'No-Shop'),
    ('non-solicitation',         'No-Shop'),
    ('fiduciary',                'Fiduciary Exception'),
    ('operating covenant',       'Operating Covenants'),
    ('material adverse effect',  'Material Adverse Effect'),
    ('mae',                      'Material Adverse Effect'),
    ('antitrust',                'Antitrust Efforts Standard'),
    ('specific performance',     'Specific Performance'),
    ('superior offer',           'Superior Offer Definition'),
    ('intervening event',        'Intervening Event Definition'),
    ('matching right',           'Matching Rights'),
    ('tail period',              'Tail Period'),
    ('expense reimbursement',    'Expense Reimbursement'),
    ('board recommendation',     'Board Recommendation Change'),
    ('change of recommendation', 'Board Recommendation Change'),
    ('change in recommendation', 'Board Recommendation Change'),
]


# ── MAUD Processor ────────────────────────────────────────────────────────
class MAUDProcessor:
    def load_and_process(self, cache_dir=MAUD_DIR) -> pd.DataFrame:
        logger.info('Loading MAUD dataset from HuggingFace...')
        dataset = load_dataset(MAUD_DATASET_ID, cache_dir=str(cache_dir))

        records = []
        skipped = 0

        for split_name in ('train', 'validation', 'test'):
            if split_name not in dataset:
                continue
            for item in tqdm(dataset[split_name], desc=f'MAUD {split_name}'):
                text = (item.get('text') or '').strip()
                if not text or len(text.split()) < CLAUSE_MIN_TOKENS:
                    skipped += 1
                    continue

                question     = (item.get('question') or '').strip().lower()
                unified_type = next(
                    (ut for kw, ut in _MAUD_QUESTION_MAP if kw in question), None
                )
                if unified_type is None:
                    skipped += 1
                    continue

                category    = (item.get('category') or 'MAUD').strip()
                contract_id = hashlib.md5(f'MAUD_{category}_{question}'.encode()).hexdigest()[:12]
                clause_id   = hashlib.md5(f'MAUD{text}'.encode()).hexdigest()[:16]
                records.append({
                    'contract_id':  contract_id,
                    'clause_id':    clause_id,
                    'clause_text':  text,
                    'clause_types': [unified_type],
                    'party_a': '', 'party_b': '',
                    'source': 'MAUD',
                })

        df = pd.DataFrame(records).drop_duplicates(subset='clause_id').reset_index(drop=True)
        logger.info(f'MAUD: {len(df)} clauses retained, {skipped} skipped')
        return df

print('LEDGARProcessor and MAUDProcessor defined.')

In [ ]:
# ── DataStore ─────────────────────────────────────────────────────────────
class DataStore:
    def __init__(self):
        create_tables()

    def save_contract(self, contract_id, source, filename):
        with SessionLocal() as s:
            if s.get(Contract, contract_id) is None:
                s.add(Contract(contract_id=contract_id, filename=filename, source=source))
                s.commit()

    def save_clauses(self, df: pd.DataFrame) -> int:
        inserted = 0
        with SessionLocal() as s:
            for _, row in tqdm(df.iterrows(), total=len(df), desc='Saving clauses'):
                if s.get(Clause, row['clause_id']) is not None:
                    continue
                ct = row.get('clause_types', [])
                ct_str = '|'.join(ct) if isinstance(ct, list) else str(ct)
                s.add(Clause(
                    clause_id=row['clause_id'], contract_id=row['contract_id'],
                    clause_text=row['clause_text'], clause_type=ct_str,
                    party_a=row.get('party_a', ''), party_b=row.get('party_b', ''),
                    source=row.get('source', ''),
                ))
                inserted += 1
            s.commit()
        logger.info(f'Saved {inserted} new clauses')
        return inserted

    def load_all_clauses(self) -> pd.DataFrame:
        with SessionLocal() as s:
            rows = s.execute(sa_text('SELECT * FROM clauses')).fetchall()
            if not rows:
                return pd.DataFrame()
            return pd.DataFrame(rows, columns=rows[0]._fields)


# ── DataSplitter ──────────────────────────────────────────────────────────
class DataSplitter:
    def split(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        df = df.copy()
        df['primary_type'] = df['clause_type'].apply(
            lambda x: x.split('|')[0] if isinstance(x, str) and x else 'Unknown'
        )
        counts      = df['primary_type'].value_counts()
        valid_types = counts[counts >= 10].index
        df_filtered = df[df['primary_type'].isin(valid_types)].copy()
        df_rare     = df[~df['primary_type'].isin(valid_types)].copy()

        if df_filtered.empty:
            raise RuntimeError('No classes have enough samples for stratified splitting.')

        train_df, temp_df = train_test_split(
            df_filtered, test_size=(1 - TRAIN_RATIO),
            stratify=df_filtered['primary_type'], random_state=RANDOM_SEED,
        )
        temp_counts      = temp_df['primary_type'].value_counts()
        temp_valid_types = temp_counts[temp_counts >= 2].index
        temp_stratified  = temp_df[temp_df['primary_type'].isin(temp_valid_types)].copy()
        temp_rare        = temp_df[~temp_df['primary_type'].isin(temp_valid_types)].copy()
        relative_val     = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

        if temp_stratified.empty:
            val_df  = pd.DataFrame(columns=df.columns)
            test_df = pd.DataFrame(columns=df.columns)
            train_df = pd.concat([train_df, df_rare, temp_rare], ignore_index=True)
        else:
            val_df, test_df = train_test_split(
                temp_stratified, test_size=(1 - relative_val),
                stratify=temp_stratified['primary_type'], random_state=RANDOM_SEED,
            )
            train_df = pd.concat([train_df, df_rare, temp_rare], ignore_index=True)

        # Undersample dominant classes in training set only (val/test untouched)
        cap = UNDERSAMPLE_MAX_PER_CLASS
        if cap is not None:
            before_us = len(train_df)
            train_df = (
                train_df
                .groupby('primary_type', group_keys=False)
                .apply(lambda g: g.sample(n=min(len(g), cap), random_state=RANDOM_SEED))
                .sample(frac=1, random_state=RANDOM_SEED)
                .reset_index(drop=True)
            )
            logger.info(f'Undersampling (cap={cap}/class): {before_us} -> {len(train_df)} training rows')

        logger.info(f'Split — Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
        return train_df, val_df, test_df

    def save_splits(self, train_df, val_df, test_df):
        train_df.to_csv(PROCESSED_DIR / 'train.csv', index=False)
        val_df.to_csv(  PROCESSED_DIR / 'val.csv',   index=False)
        test_df.to_csv( PROCESSED_DIR / 'test.csv',  index=False)
        logger.info(f'Splits saved to {PROCESSED_DIR}')

print('DataStore and DataSplitter defined.')

## Step 6 — Download & Process CUAD Dataset

⏱ *This step downloads ~1.5 GB from HuggingFace. It may take 10–20 minutes depending on Colab speed. The dataset is cached to Drive so subsequent runs are instant.*

In [ ]:
cuad   = CUADProcessor()
store  = DataStore()

df = cuad.load_and_process()

for contract_id in df['contract_id'].unique():
    store.save_contract(contract_id=contract_id, source='CUAD', filename=contract_id)

store.save_clauses(df)

print(f'\nCUAD pipeline complete.')
print(f'  Contracts : {df["contract_id"].nunique()}')
print(f'  Clauses   : {len(df)}')
print(f'  DB saved  : {DB_PATH}')

## Step 9 — Generate Train / Val / Test Splits

## Step 7 — Download & Process LEDGAR Dataset

⏱ *~80,000 provisions across 100 provision types. Download takes 5–10 minutes; cached to Drive for subsequent runs.*

In [ ]:
ledgar = LEDGARProcessor()
df_ledgar = ledgar.load_and_process()

for contract_id in df_ledgar['contract_id'].unique():
    store.save_contract(contract_id=contract_id, source='LEDGAR', filename=contract_id)

store.save_clauses(df_ledgar)

print(f'\nLEDGAR pipeline complete.')
print(f'  Clauses   : {len(df_ledgar)}')
print(f'  Types     : {df_ledgar["clause_types"].explode().nunique()} unique clause types')
print(f'  DB path   : {DB_PATH}')

## Step 8 — Download & Process MAUD Dataset

⏱ *~39,000 M&A deal excerpts across 14 deal-point types. Download takes 2–5 minutes.*

In [ ]:
maud = MAUDProcessor()
df_maud = maud.load_and_process()

for contract_id in df_maud['contract_id'].unique():
    store.save_contract(contract_id=contract_id, source='MAUD', filename=contract_id)

store.save_clauses(df_maud)

print(f'\nMAUD pipeline complete.')
print(f'  Clauses   : {len(df_maud)}')
print(f'  Types     : {df_maud["clause_types"].explode().nunique()} unique clause types')
print(f'  DB path   : {DB_PATH}')

# Combined summary across all three datasets
total = len(df) + len(df_ledgar) + len(df_maud)
print(f'\nTotal clauses in DB across all datasets: ~{total:,}')

In [ ]:
all_clauses = store.load_all_clauses()
print(f'Total clauses in DB: {len(all_clauses)}')

splitter = DataSplitter()
train_df, val_df, test_df = splitter.split(all_clauses)
splitter.save_splits(train_df, val_df, test_df)

print(f'\nSplits saved to {PROCESSED_DIR}')
print(f'  train.csv : {len(train_df)} rows')
print(f'  val.csv   : {len(val_df)} rows')
print(f'  test.csv  : {len(test_df)} rows')

## Step 10 — Download Outputs

Downloads all 4 output files to your **local machine**.

Place them at: `contract-intelligence-platform/data/processed/`

In [ ]:
from google.colab import files

outputs = [
    PROCESSED_DIR / 'contracts.db',
    PROCESSED_DIR / 'train.csv',
    PROCESSED_DIR / 'val.csv',
    PROCESSED_DIR / 'test.csv',
]

for fpath in outputs:
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1_048_576
        print(f'Downloading {fpath.name} ({size_mb:.1f} MB)...')
        files.download(str(fpath))
    else:
        print(f'WARNING: {fpath.name} not found — check previous steps for errors')

print('\nDone. Place downloaded files into:')
print('  contract-intelligence-platform/data/processed/')

## (Optional) Verify outputs

Quick sanity check on the downloaded data.

In [ ]:
for name in ['train.csv', 'val.csv', 'test.csv']:
    p = PROCESSED_DIR / name
    if p.exists():
        d = pd.read_csv(p)
        print(f'{name}: {len(d)} rows, {d["clause_type"].nunique()} unique clause types')

print('\nTop 10 clause types in train set:')
train = pd.read_csv(PROCESSED_DIR / 'train.csv')
print(train['clause_type'].value_counts().head(10).to_string())